# Writing and Running Your First Scripts

**Topic:** `1.8` &nbsp;|&nbsp; **Module 1: Getting Started with Python**

*Turning a module into a program: the main guard, sys.argv, exit codes and the three-channel contract between a script and whoever started it.*

> *Course:* Python Master Course &nbsp;|&nbsp; **Project:** ROBO-X (M1)

### Navigate

* [Lesson](lesson.ipynb) &nbsp;|&nbsp; [Exercises](exercises.ipynb) &nbsp;|&nbsp; [Solutions](solution.ipynb) &nbsp;|&nbsp; [Research](research.ipynb) &nbsp;|&nbsp; [Quiz](quiz.ipynb) &nbsp;|&nbsp; [Mini-project](mini_project.ipynb) &nbsp;|&nbsp; [Robotics challenge](robotics_challenge.ipynb)

Module index: [01_getting_started/README.md](../README.md)

## 1. Why This Topic Matters

A fleet supervisor launches robot services as child processes and decides what happens next from the exit code alone. Getting that contract right is the difference between a tool that composes with a system and one that is run by hand and hoped over.

## 2. Learning Objectives

By the end of this topic you will be able to:

1. Explain how __name__ distinguishes a module from a program.
2. Guard side effects so importing a script cannot start a robot.
3. Pass sys.argv[1:] into logic instead of indexing it inline.
4. Return an exit status and call sys.exit exactly once.
5. Keep results on stdout, diagnostics on stderr, and decisions in the exit code.
6. Test a tool without a shell by injecting argv and the streams.

## 3. Prerequisites

This topic assumes you already have:

- Topic 1.7 Basic Input/Output

## 4. Mental Model

**Mental model: a file has two personalities and one switch.** Imported, it is a library that offers things and does nothing. Executed, it is a program that acts. `__name__` is the switch, and the main guard is the switch's housing. Everything else in this topic - argv, exit codes, streams - is the contract between that program and whoever started it.

|  | Imported | Executed directly |
| --- | --- | --- |
| `__name__` | the module's own name | `'__main__'` |
| Top-level code | runs | runs |
| Guarded code | skipped | runs |
| Exit code | irrelevant | handed to the shell |
| Reusable | yes | no |

## 5. Terminology

Vocabulary used consistently throughout the course.

| Term | Meaning |
| --- | --- |
| Script | A file executed directly by the interpreter as a program. |
| Module | Any importable Python file, including a script. |
| `__name__` | The name a module is known by; `'__main__'` when executed directly. |
| Main guard | The `if __name__ == '__main__':` block protecting side effects. |
| Shebang | The first line naming the interpreter for the shell. |
| `sys.argv` | The list of command-line arguments, program path first. |
| Exit code | The integer status a process hands back to the shell. |
| `SystemExit` | The exception that ends a process with a status. |
| argv[0] | The path used to invoke the script, not an argument. |
| Subprocess | A child process, observed through its streams and return code. |

## 6. Conceptual Explanation

Every Python file is a **module**. It only becomes a **program** when the interpreter runs it as the top-level script, and the interpreter tells you which case you are in through the module-level name `__name__`. Importing a file sets it to the module's own name; executing it directly sets it to `"__main__"`. The `if __name__ == "__main__":` guard is the idiom that uses that distinction.

In [ ]:
def demo() -> str:
    return f'running as {__name__}'


print(demo())

if __name__ == '__main__':
    print('this block only runs when executed directly')

Without the guard, importing a module runs its top-level code. For a library that is merely wasteful; for a script that starts a robot, it is dangerous - a test that imports the module would launch the hardware. The guard is therefore not a stylistic preference but a safety boundary.

In [ ]:
import sys

print('argv as seen by this program:')
for index, value in enumerate(sys.argv):
    print(f'  [{index}] {value!r}')

print('the name this module is running as:', __name__)

> **sys.argv[0] is the program, not a parameter**
>
> The first element is the path used to invoke the script, and the rest are the arguments. Slicing from index 1 is the single most common early mistake, and it usually shows up as an off-by-one that only appears with zero arguments.

## 7. Formal Theory

Running a script has four stages: the interpreter starts, the file is compiled to bytecode, the module is executed top to bottom with `__name__` set to `"__main__"`, and the resulting exit status is handed back to the shell. The exit status comes from the value passed to `SystemExit`, or from an uncaught exception, and defaults to zero.

```text
python script.py args...  ->  exit 0 on success, non-zero on SystemExit(n) or an uncaught error
```

The shell is part of the contract. A script is often launched by another program rather than a person, and that caller can only see three things: the exit code, stdout, and stderr. Everything a caller needs to branch on must appear in one of those three, which is why the exit code and the stream discipline from Topic 1.7 matter here.

In [ ]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, '-c', 'import sys; sys.exit(3)'],
    capture_output=True,
    text=True,
)
print('returncode:', result.returncode)
print('succeeded :', result.returncode == 0)

boom = subprocess.run(
    [sys.executable, '-c', 'raise SystemExit("bad config")'],
    capture_output=True,
    text=True,
)
print('message   :', boom.stderr.strip())

| Stage | What happens | How to observe it |
| --- | --- | --- |
| Start | The interpreter initialises | `sys.executable` |
| Compile | Source becomes bytecode | A SyntaxError stops here |
| Execute | Top-level statements run | `__name__` is `'__main__'` |
| Exit | A status reaches the shell | The process return code |

## 8. Syntax

A runnable script has three parts: an optional shebang so the shell knows the interpreter, a docstring, and a guarded entry point.

In [ ]:
#!/usr/bin/env python3
"""A minimal, well-formed command-line script."""
from __future__ import annotations

import sys


def main(argv: list[str]) -> int:
    """Return the process exit status."""
    if len(argv) < 2:
        print('usage: script.py <name>', file=sys.stderr)
        return 2
    print(f'hello, {argv[1]}')
    return 0


if __name__ == '__main__':
    main(sys.argv)

> **Why this cell calls main() instead of sys.exit(main())**
>
> A real script ends with `sys.exit(main(sys.argv))` so the shell receives the status. Running that inside a notebook would terminate the kernel, so the cell calls main() directly. Use the SystemExit form in the scripts you write later.

The anti-pattern:

```text
import sys
robot = SimulatedRobot()
robot.move(10)
print('done')

# everything above runs on import, including the movement
def main():
    ...

if __name__ == '__main__':
    main()
```

> **Do not call sys.exit deep inside logic**
>
> SystemExit is an exception: raising it from a helper unwinds the whole stack. Return a status from main and exit once, at the bottom, where the intent is clear.

## 9. Basic Examples

**Example 1 - separating the program name from the arguments.**

In [ ]:
import sys


def arguments(argv: list[str]) -> list[str]:
    """Drop argv[0], which is the program path, not an argument."""
    return list(argv[1:])


print('full   :', sys.argv)
print('args   :', arguments(sys.argv))
print('joined :', ' '.join(arguments(sys.argv)))

**Example 2 - compiling a source string before running it.**

In [ ]:
source = 'value = 6 * 7\nresult = value\n'
namespace: dict = {}
code = compile(source, '<demo>', 'exec')
exec(code, namespace)
print('compiled and executed ->', namespace['result'])

## 10. Intermediate Examples

Running a child process is how one script launches another, and it is also the cleanest way to see your own exit-code contract from the outside.

In [ ]:
import subprocess
import sys


def run(script_args: list[str]) -> tuple[int, str]:
    result = subprocess.run(
        [sys.executable, '-c', 'import sys; print(sys.argv[1:])'] + script_args,
        capture_output=True,
        text=True,
        timeout=5,
    )
    return result.returncode, result.stdout.strip()


print(run(['a', 'b']))
print(run([]))

## 11. Advanced Examples

`runpy` executes a module by name with control over whether it sees itself as `__main__`, which is exactly what you need to test a script's entry point without launching it in a shell.

In [ ]:
import runpy
import sys

captured = {}
original = sys.argv
sys.argv = ['demo.py']
try:
    runpy.run_path('01_getting_started/08_writing_running_first_scripts/README.md',
                  run_name='__demo__')
except (FileNotFoundError, IsADirectoryError, SyntaxError) as exc:
    captured['error'] = type(exc).__name__
finally:
    sys.argv = original

print('run_name controls __name__; failures surface as:', captured)

## 12. Code Walkthrough

Below is a complete, runnable command-line tool. It is the shape every later module's exercises eventually take: parse, act, return a status.

In [ ]:
"""roboctl.py - a small fleet control tool."""
from __future__ import annotations

import sys

EXIT_OK, EXIT_UNKNOWN, EXIT_USAGE = 0, 1, 2
COMMANDS = ('status', 'stop', 'help')


def render_status() -> str:
    return 'rover-01:ACTIVE:48.0'


def run(argv: list[str], out=sys.stdout, errors=sys.stderr) -> int:
    """Dispatch one command. Writes only to the streams it is given."""
    if not argv:
        print(f'usage: roboctl <{"|".join(COMMANDS)}>', file=errors)
        return EXIT_USAGE
    command = argv[0]
    if command == 'help':
        print(f'usage: roboctl <{"|".join(COMMANDS)}>', file=out)
        return EXIT_OK
    if command == 'status':
        print(render_status(), file=out)
        return EXIT_OK
    if command == 'stop':
        print('stopped', file=out)
        return EXIT_OK
    print(f'unknown command: {command!r}', file=errors)
    return EXIT_UNKNOWN


def main() -> int:
    return run(sys.argv[1:])


if __name__ == '__main__':
    main()

Three properties make this testable. `run` takes argv and both streams as parameters, so no test needs a subprocess. `render_status` is a pure function that returns a line, so its output can be asserted directly. And the exit code is returned rather than raised, so the same function serves a shell and a test.

> **Why the guard calls main() and not sys.exit(main())**
>
> A real script ends with `sys.exit(main())` so the shell receives the status. Running that in a notebook would terminate the kernel, so this cell calls `main()` and simply prints what it returned. Use the SystemExit form in the command-line tools you build later in the course.

## 13. Line-by-Line Explanation

1. The exit codes are named constants, so a caller and a test refer to the same meaning rather than to a bare integer.
2. COMMANDS is a tuple used only for the usage message, keeping the accepted set visible in one place.
3. render_status returns a string and prints nothing, which is what makes the output assertable without capturing a stream.
4. run takes argv and both streams as parameters, so a test supplies them directly instead of spawning a process.
5. argv[0] is the subcommand because the caller already passed sys.argv[1:] - this is where the off-by-one is avoided.
6. Each branch returns its own status, so the caller never has to infer success from printed text.
7. main is a two-line adapter whose only job is to turn process state into arguments for run.
8. The guard at the bottom converts that returned status into the process exit code, in exactly one place.

## 14. Common Mistakes

**Mistake 1 - forgetting the main guard.**

```text
import sys
print('starting')
sys.exit(main())      # runs on import, killing the importing process

if __name__ == '__main__':
    sys.exit(main())
```

**Mistake 2 - treating argv[0] as an argument.**

```text
name = sys.argv[0]     # this is the program path
name = sys.argv[1]     # only if an argument was actually passed

args = sys.argv[1:]    # the correct general form
```

**Mistake 3 - calling sys.exit inside a helper.**

```text
def load(path):
    if not exists(path):
        sys.exit(1)      # unwinds the whole stack
    return open(path)

def load(path):
    if not exists(path):
        raise FileNotFoundError(path)
```

**Mistake 4 - printing the only error message.**

```text
print('error: bad config')          # a caller cannot detect this
sys.exit(2)                        # and the status is lost

print('error: bad config', file=sys.stderr)
sys.exit(2)                        # now both channels say it
```

## 15. Debugging Techniques

The first question about a script that behaves oddly when launched is whether it behaves differently when imported. That single check distinguishes a main guard bug from everything else.

In [ ]:
def probe() -> dict:
    """Return what this module believes about its own execution."""
    import sys
    return {'name': __name__, 'argv0': sys.argv[0]}


print('as executed:', probe())
print('argv0 is the path that started this program')

To see a script's exit code from another program, run it as a subprocess rather than importing it - that is the only way to observe the contract a shell would see.

In [ ]:
import subprocess
import sys

for code in ('0', '2', '7'):
    result = subprocess.run(
        [sys.executable, '-c', f'import sys; sys.exit({code})'],
        capture_output=True,
        text=True,
    )
    print(f'asked for {code} -> shell sees {result.returncode}')

## 16. Best Practices

- Guard the entry point with `if __name__ == '__main__':` in every script.
- Pass `sys.argv[1:]` into a function; never index it inside the logic.
- Return an exit status from main and call sys.exit exactly once.
- Use named exit-code constants so callers and tests share a meaning.
- Take streams as parameters so the tool can be tested without a subprocess.
- Report errors on stderr *and* return a non-zero status.
- Start with a shebang and a docstring so the file is self-describing.
- Keep a shebang as `#!/usr/bin/env python3` so it works across machines.

## 17. Performance Considerations

Interpreter start-up is the fixed cost of every script, and it is larger than most beginners expect: importing the interpreter and reaching your first statement typically costs tens of milliseconds. For a script invoked once per second by a supervisor, that is negligible. For one invoked in a tight loop, or on a control path, it is not - and the answer is a resident service or a compiled tool, not a faster script.

Within the script, the usual cost centres apply: importing a large library at module level when the script may exit early, and building a large data structure before discovering there is nothing to do. Deferring an expensive import into the branch that needs it, and checking for empty input before building output, are the two changes that pay off most often in small tools.

> **Measure start-up, not micro-steps**
>
> Time `python -c pass` to establish the floor, then time your script. The difference is what your code costs, and it is the only number worth discussing.

## 18. Pythonic Approaches

Scanning argv: a manual index loop versus membership tests.

In [ ]:
args = ['--verbose', '--name', 'rover-01', '--count', '3']

# before: manual index arithmetic
verbose = False
name = None
index = 0
while index < len(args):
    if args[index] == '--verbose':
        verbose = True
        index += 1
    elif args[index] == '--name' and index + 1 < len(args):
        name = args[index + 1]
        index += 2
    else:
        index += 1

# after: say what you are looking for, and let the language scan
verbose = '--verbose' in args
name = args[args.index('--name') + 1] if '--name' in args else None
print('verbose:', verbose, '| name:', name)

## 19. Robotics Connection

A fleet supervisor launches robot services exactly the way a shell does: as child processes whose exit status decides whether the robot is allowed to proceed. That is why a correct exit code, and a non-zero one on failure, is a safety feature rather than a formality.

In [ ]:
import subprocess
import sys
from pathlib import Path

here = Path.cwd()
root = next((p for p in [here, *here.parents] if (p / 'shared').is_dir()), None)
if root is not None:
    sys.path.insert(0, str(root))

from shared.robo_x_sim import SimulatedRobot  # noqa: E402

robot = SimulatedRobot(name='rover-01', battery_wh=48.0)
state = robot.status()
print(f"{state['name']}:{state['battery_pct']:.1f}")

# a supervisor sees only this: the status and the two streams
failed = subprocess.run(
    [sys.executable, '-c', 'import sys; sys.exit(1)'],
    capture_output=True,
)
print('child failed ->', failed.returncode != 0)

## 20. Engineering Example

A dispatch function that is pure with respect to the process: everything it needs arrives as an argument.

In [ ]:
import sys

EXIT_OK, EXIT_UNKNOWN, EXIT_USAGE = 0, 1, 2
COMMANDS = ('status', 'stop', 'help')


def dispatch(argv, out=sys.stdout, errors=sys.stderr) -> int:
    """Run one command; return the exit status a shell would observe."""
    if not argv:
        print(f'usage: roboctl <{"|".join(COMMANDS)}>', file=errors)
        return EXIT_USAGE
    command = argv[0]
    if command not in COMMANDS:
        print(f'unknown command: {command!r}', file=errors)
        return EXIT_UNKNOWN
    if command == 'help':
        print(f'usage: roboctl <{"|".join(COMMANDS)}>', file=out)
    else:
        print(f'{command}: ok', file=out)
    return EXIT_OK

## 21. Guided Practice

1. Write a module that prints something at import time, then add the main guard and import it again to see the difference.
2. Write `arguments(argv)` that returns everything after the program name, and test it with an empty list.
3. Run a child process that exits with 3 and confirm the return code you observe.
4. Write `dispatch(argv)` returning 0, 1 and 2 for success, unknown command and missing argument.

In [ ]:
import subprocess
import sys


def arguments(argv):
    return list(argv[1:])


print('args of []  ->', arguments([]))
print('args of [a] ->', arguments(['prog', 'a']))

child = subprocess.run(
    [sys.executable, '-c', 'import sys; sys.exit(3)'],
    capture_output=True,
)
print('child exit code ->', child.returncode)

print('this module is running as', __name__)

## 22. Knowledge Checks

Answer these before moving on. Full answers are in [solution.ipynb](solution.ipynb).

- **Q1.** What is the value of `__name__` when a file is executed directly?
- **Q1.** What can a shell observe about a running program?
- **Q1.** A fleet supervisor launches a service and checks its return code. What does a return code of 2 most likely mean here?
- **Q1.** Which technique proves a module is import-safe?

## 23. Summary

Every Python file is a module; it becomes a program only when executed directly, and `__name__` is how the interpreter tells the two cases apart. The main guard is therefore not ceremony. Without it, importing a file runs its top-level code, which for a robotics script means a test that imports the module launches the robot.

The second half of the topic is the contract with whoever started the process. That caller - a shell, a supervisor, a CI job - can see only three things: the exit code, stdout and stderr. So every decision the caller needs to make must appear in one of them. Return a status from main, call `sys.exit` once at the bottom, print results to stdout and problems to stderr, and your tool composes with everything else on the machine.

The practical habit is to keep the process at arm's length. Write the real work as a function that takes its arguments and its streams as parameters, and let a two-line `main` translate process state into those arguments. That is why the tool in the walkthrough can be tested with a StringIO and no subprocess, while still behaving exactly as it would in a shell.

## 24. Key Takeaways

- Every file is a module; only direct execution makes it a program.
- `__name__` is `'__main__'` only when executed directly.
- Guard side effects so importing a script cannot launch it.
- `sys.argv[0]` is the program path, not an argument.
- Pass `sys.argv[1:]` into a function rather than indexing it inline.
- Return an exit status from main and call sys.exit once.
- Results go to stdout, problems to stderr, status to the exit code.
- Take streams and argv as parameters so the tool is testable.

## 25. Further Exploration

- [`if __name__ == '__main__'`: a gotcha](https://stackoverflow.com/questions/419163/what-does-if-name-main-do)
- [sys.argv and the command line](https://docs.python.org/3/library/sys.html#sys.argv)
- [SystemExit](https://docs.python.org/3/library/exceptions.html#SystemExit)
- [subprocess - running child processes](https://docs.python.org/3/library/subprocess.html)
- [PEP 8 - shebangs and module structure](https://peps.python.org/pep-0008/)